#Part 1 – Downloading and Installing the required software

### 1. Environment Setup 🛠️
Before we start docking, we need to install our computational biology toolkit and download the docking engine (**AutoDock Vina**).

* **Python Libraries:** `rdkit` & `meeko` (for ligands), `biopython` & `gemmi` (for protein handling), `pdb2pqr` (charge assignment), and `py3Dmol` (3D visualization inside the notebook).
* **Binaries:** AutoDock Vina v1.2.7 and its splitting utility.

In [7]:
# 1. Install required Python packages
!pip -q install biopython py3Dmol rdkit meeko gemmi pdb2pqr
# 1. Install OpenBabel
!apt-get -qq install openbabel -y

Selecting previously unselected package libboost-iostreams1.83.0:amd64.
(Reading database ... 122797 files and directories currently installed.)
Preparing to unpack .../libboost-iostreams1.83.0_1.83.0-2.1ubuntu3.2_amd64.deb ...
Unpacking libboost-iostreams1.83.0:amd64 (1.83.0-2.1ubuntu3.2) ...
Selecting previously unselected package libinchi1.
Preparing to unpack .../libinchi1_1.03+dfsg-4build1_amd64.deb ...
Unpacking libinchi1 (1.03+dfsg-4build1) ...
Selecting previously unselected package libmaeparser1:amd64.
Preparing to unpack .../libmaeparser1_1.3.1-1build1_amd64.deb ...
Unpacking libmaeparser1:amd64 (1.3.1-1build1) ...
Selecting previously unselected package libopenbabel7.
Preparing to unpack .../libopenbabel7_3.1.1+dfsg-9ubuntu5_amd64.deb ...
Unpacking libopenbabel7 (3.1.1+dfsg-9ubuntu5) ...
Selecting previously unselected package openbabel.
Preparing to unpack .../openbabel_3.1.1+dfsg-9ubuntu5_amd64.deb ...
Unpacking openbabel (3.1.1+dfsg-9ubuntu5) ...
Setting up libboost-iostr

In [8]:
%%bash
# 2. Set up workspace and download AutoDock Vina binaries
mkdir -p /content/vina
cd /content/vina

# Download Vina and make it executable
wget -q -nc https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.7/vina_1.2.7_linux_x86_64
chmod +x vina_1.2.7_linux_x86_64

# Download vina_split (useful for separating multiple docking poses later)
wget -q -nc https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.7/vina_split_1.2.7_linux_x86_64
chmod +x vina_split_1.2.7_linux_x86_64

# Create symlinks to /usr/local/bin so we can call 'vina' directly from anywhere
ln -sf /content/vina/vina_1.2.7_linux_x86_64 /usr/local/bin/vina
ln -sf /content/vina/vina_split_1.2.7_linux_x86_64 /usr/local/bin/vina_split

# Quick check to verify installation
vina --version

AutoDock Vina v1.2.7


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [9]:
# Import all core libraries to make sure they are loaded properly
import Bio
import gemmi
import meeko
import py3Dmol
import rdkit

print("All modern Python libraries are successfully imported!")
# Check that the AutoDock Vina binary is accessible via the command line
!vina --version

All modern Python libraries are successfully imported!
AutoDock Vina v1.2.7


#Part 2 – Downloading and Preparing the Receptor for AutoDock

### 2. Preparing the Target Protein
The first step in any molecular docking protocol is obtaining a 3D structure of the biological target. Typically, we use experimentally resolved structures (X-ray, Cryo-EM, or NMR) deposited in the [Protein Data Bank (RCSB PDB)](https://www.rcsb.org/).

In this tutorial, we will use the **HIV-1 protease** complex (PDB ID: **1HSG**), an X-ray solved homodimer co-crystallized with the inhibitor Indinavir (MK1).

In [10]:
import os
from Bio.PDB import PDBList
pdbid = ['1hsg']
pdbl = PDBList()
for s in pdbid:
    pdbl.retrieve_pdb_file(s, pdir='.', file_format="pdb", overwrite=True)
    os.rename(f"pdb{s}.ent", f"{s}.pdb")

<figure>
<center>
<img src='https://raw.githubusercontent.com/pb3lab/ibm3202/master/images/docking_03.png' />
<figcaption>FIGURE 3. Cartoon representation of HIV-2 protease dimer (PDB accession ID 1HSG), with its N-to-C-terminal residues colored from blue to red in rainbow spectrum, showing the crystallographic waters as red spheres.</i></figcaption></center>
</figure>

### 2.2 Structure Cleaning: Extracting the Apo-Protein
PDB files contain multiple record types:
* **`ATOM`**: Standard residues belonging to the protein chain.
* **`HETATM`**: Non-polymer entities (water molecules, ions, and the bound ligand MK1).
* **`TER`**: Chain termination boundary. Retaining this is critical for HIV-1 protease (a homodimer of chains A and B) to prevent artificial bonds across chains.

We parse `1hsg.pdb` to retain **only** `ATOM` and `TER` records, isolating a clean apo-receptor without crystallographic waters or the native ligand.

In [11]:
from pathlib import Path

# Setup clean working directory
docking_dir = Path("/content/single-dock")
docking_dir.mkdir(parents=True, exist_ok=True)

input_pdb = Path("1hsg.pdb")
output_pdb = docking_dir / "1hsg_prot.pdb"

# Filter line-by-line for clean protein-only coordinates
with open(input_pdb, "r") as infile, open(output_pdb, "w") as outfile:
    for line in infile:
        if line.startswith("ATOM"):
            outfile.write(line)
        elif line.startswith("TER"):
            outfile.write("TER\n")
    outfile.write("END\n")

print(f"Clean apo-protein saved to: {output_pdb}")

Clean apo-protein saved to: /content/single-dock/1hsg_prot.pdb


### 2.3 Protonation & PDBQT Conversion
AutoDock Vina requires coordinate files in **PDBQT** format (which includes Partial Charges **Q** and Atom Types **T**).

We prepare our receptor in two simple stages:
1. **PDB2PQR:** Adds missing hydrogen atoms, predicts protonation states at physiological pH (7.4) using **PROPKA**, and assigns **AMBER** force-field charges (generating a `.pqr` file).
2. **OpenBabel:** Converts the `.pqr` structure into the final rigid `.pdbqt` format required by Vina.

In [12]:
from pathlib import Path

singlepath = Path("/content/single-dock")
prot_pdb = singlepath / "1hsg_prot.pdb"
prot_pqr = singlepath / "1hsg_prot.pqr"
prot_pdbqt = singlepath / "1hsg_prot.pdbqt"

# 1. Parameterize receptor at pH 7.4 using pdb2pqr
!pdb2pqr --ff AMBER --keep-chain --titration-state-method propka --with-ph 7.4 {prot_pdb} {prot_pqr}

# 2. Install OpenBabel so Linux recognizes the 'obabel' command
!apt-get -qq install openbabel -y

# 3. Convert PQR to PDBQT directly using OpenBabel (-xr keeps the receptor rigid)
!obabel -ipqr {prot_pqr} -opdbqt -O {prot_pdbqt} -xr

print("Receptor PDBQT created successfully!")

Streaming output truncated to the last 5000 lines.
 charge : 29.318
 radius : 0.8076
 charge : 29.04
 radius : -0.8627
 charge : 29.994
 radius : -0.8627
 charge : 25.453
 radius : 0.2747
 charge : 26.158
 radius : 0.156
 charge : 25.935
 radius : 0.0327
 charge : 24.951
 radius : 0.0327
 charge : 26.688
 radius : 0.0285
 charge : 27.24
 radius : 0.0285
 charge : 27.829
 radius : 0.0687
 charge : 28.764
 radius : 0.0687
 charge : 29.184
 radius : 0.3456
 charge : 28.559
 radius : 0.4478
 charge : 29.319
 radius : 0.4478
 charge : 30.227
 radius : 0.4478
 charge : 30.268
 radius : 0.4478
 charge : 23.834
 radius : -0.4157
 charge : 22.533
 radius : -0.0275
 charge : 21.959
 radius : 0.5973
 charge : 22.694
 radius : -0.5679
 charge : 22.642
 radius : -0.005
 charge : 23.787
 radius : -0.1415
 charge : 25.041
 radius : -0.1638
 charge : 23.857
 radius : 0.1243
 charge : 25.884
 radius : -0.3418
 charge : 25.193
 radius : 0.138
 charge : 22.909
 radius : -0.2387
 charge : 25.618
 radius :


-------
### 💡 Concept Check: Why Do We Prepare the Receptor This Way?

Before moving to the ligand, let's understand the biophysical rationale behind our preparation steps:

#### 1. Why is it important to add hydrogens for docking?
X-ray crystallography rarely resolves hydrogen atoms because of their low electron density. However, hydrogens are essential for:
* Forming directional **hydrogen bonds** between the ligand and receptor.
* Setting the correct **steric boundaries** (Van der Waals radii) inside the binding pocket to prevent clashes.

#### 2. Why do we only add *polar* hydrogens?
AutoDock Vina uses a **United-Atom Model** for efficiency:
* **Non-polar hydrogens** (like those on $-CH_2-$ or $-CH_3$) are computationally "merged" into the carbon atom they are bonded to, reducing calculation overhead.
* **Polar hydrogens** (bonded to electronegative atoms like $O$ and $N$) are explicitly retained because they directly participate in hydrogen bonding and electrostatic interactions.

#### 3. If partial charges are largely ignored by Vina's scoring function, how do different preparation methods affect docking results?
AutoDock Vina relies heavily on steric fit (shape complementarity), hydrophobic contacts, and hydrogen bonding rather than explicit Coulombic electrostatics. However, the preparation strategy (and tools like PDB2PQR) directly impacts:
* **Hydrogen placement and protonation states:** Predicting whether a Histidine or Aspartate is protonated or neutral changes which hydrogen-bond donors and acceptors are available.
* **Rotamer orientation of polar groups:** Flipping Ser/Thr/Tyr hydroxyls or Asn/Gln amides drastically alters the local shape and H-bonding network of the binding site.
---

# 3. Ligand Preparation: Indinavir (MK-1)
To perform molecular docking, we need a realistic 3D model of the small-molecule inhibitor. In this benchmark experiment, we will dock **Indinavir** into the catalytic pocket of HIV-1 protease (PDB ID: **1HSG**).

#### Biological Context
**Indinavir** (marketed as *Crixivan*) is a transition-state peptidomimetic inhibitor designed to mimic the Phe-Pro peptide cleavage site. It binds to the catalytic aspartates (Asp25 / Asp25'), preventing viral polyprotein processing and arresting HIV viral maturation.

#### Chemical Representation: SMILES
We represent the ligand using **SMILES** (*Simplified Molecular-Input Line-Entry System*), a 1D ASCII text format encoding molecular graph topology:
* **Bonds:** `-` (single, usually omitted), `=` (double), `#` (triple), `:` (aromatic).
* **Aromaticity:** Lowercase symbols represent aromatic systems (e.g., `c1ccccc1` for benzene).
* **Branches:** Enclosed within parentheses `( )`.
* **Rings:** Defined by matching numerical tags (e.g., `c1...c1`).
* **Stereochemistry:** `@` / `@@` indicate tetrahedral chirality (counter-clockwise / clockwise), ensuring stereochemical fidelity.

Instead of relying on external API calls, we embed the verified canonical SMILES directly into our workflow.

In [15]:
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

# 1. Setup ligands directory using modern pathlib
ligand_dir = Path("/content/ligands")
ligand_dir.mkdir(parents=True, exist_ok=True)

# 2. Canonical SMILES for Indinavir (PDB Ligand ID: MK1 | DrugBank: DB00224)
indinavir_smiles = "CC(C)(C)NC(=O)[C@@H]1CN(CCN1C[C@@H](O)C[C@@H](Cc2ccccc2)C(=O)N[C@@H]3[C@H](O)Cc4ccccc34)Cc5cccnc5"

# 3. Sanity check: Ensure RDKit can parse the SMILES correctly
mol = Chem.MolFromSmiles(indinavir_smiles)
assert mol is not None, "Error: Invalid SMILES string!"

# 4. Save SMILES string to disk
smiles_file = ligand_dir / "indinavir.smi"
smiles_file.write_text(indinavir_smiles.strip())

# 5. Summary output
print(f" Ligand directory: {ligand_dir}")
print(f" SMILES file saved: {smiles_file.name}")
print(f" Molecular Weight: {Descriptors.MolWt(mol):.2f} g/mol")
print(f" Formula: {rdMolDescriptors.CalcMolFormula(mol)}")

 Ligand directory: /content/ligands
 SMILES file saved: indinavir.smi
 Molecular Weight: 613.80 g/mol
 Formula: C36H47N5O4


### 3.1 3D Conformer Generation & Visualization 🔬
Now we convert our 1D SMILES into a realistic 3D structure using **ETKDGv3** (which enforces realistic bond angles and rotatable torsions) followed by **MMFF94** energy minimization to resolve atomic clashes.

Use the interactive viewer below to inspect the generated 3D conformation.

In [17]:
import py3Dmol
from rdkit import Chem
from rdkit.Chem import AllChem
from ipywidgets import interact

def smiles_to_3d(smiles: str, random_seed: int = 42):
    """Convert a SMILES string into an energy-minimized 3D RDKit Mol object."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        print(" Invalid SMILES string.")
        return None

    # 1. Add explicit hydrogens needed for realistic 3D volume
    mol = Chem.AddHs(mol)

    # 2. Generate 3D coordinates using ETKDGv3
    params = AllChem.ETKDGv3()
    params.randomSeed = random_seed
    params.useRandomCoords = True
    params.maxIterations = 500  # Correct attribute name in RDKit

    if AllChem.EmbedMolecule(mol, params) != 0:
        # Fallback embedding if initial attempt fails
        AllChem.EmbedMolecule(mol, useRandomCoords=True, randomSeed=random_seed)

    # 3. Energy minimization using MMFF94 force field
    AllChem.MMFFOptimizeMolecule(mol, maxIters=1000)

    return mol

def render_molecule_3d(mol, style: str = "stick"):
    """Visualize the 3D structure interactively using py3Dmol."""
    mol_block = Chem.MolToMolBlock(mol)
    viewer = py3Dmol.view(width=450, height=400)
    viewer.addModel(mol_block, "mol")
    viewer.setStyle({style: {"colorscheme": "cyanCarbon"}})
    viewer.zoomTo()
    return viewer

# Interactive viewer widget
@interact
def view_smiles(smi=indinavir_smiles):
    mol_3d = smiles_to_3d(smi)
    if mol_3d:
        return render_molecule_3d(mol_3d).show()

interactive(children=(Text(value='CC(C)(C)NC(=O)[C@@H]1CN(CCN1C[C@@H](O)C[C@@H](Cc2ccccc2)C(=O)N[C@@H]3[C@H](O…

### 3.2 Ligand Parameterization & PDBQT Conversion
To make the ligand readable by AutoDock Vina, we use **Meeko** (the modern replacement for legacy AutoDockTools scripts).

#### What happens in this step?
* **Non-polar Hydrogens Merged:** United-atom simplification to speed up docking.
* **AutoDock Atom Types Assigned:** Ensuring aromatic carbons, nitrogens, etc., get the correct van der Waals parameters.
* **Torsion Tree Setup:** Meeko automatically identifies rotatable single bonds (defining the rigid core versus flexible branches).

In [20]:
from pathlib import Path
from meeko import MoleculePreparation
from rdkit import Chem
from rdkit.Chem import AllChem

# 1. Setup paths
singlepath = Path("/content/single-dock")
singlepath.mkdir(parents=True, exist_ok=True)
ligand_pdbqt_out = singlepath / "indinavir.pdbqt"

# 2. Build 3D Conformer using the already-defined smiles_to_3d() function
mol = smiles_to_3d(indinavir_smiles)
if mol is None:
    raise ValueError("3D conformation generation failed: invalid or problematic SMILES string.")

# 3. Parameterization & PDBQT conversion using Meeko
preparator = MoleculePreparation()
preparator.prepare(mol)

# 4. Generate and save the PDBQT content
pdbqt_string = preparator.write_pdbqt_string()
ligand_pdbqt_out.write_text(pdbqt_string)

# Quick validation
active_torsions = pdbqt_string.count("BRANCH")
print(f" Ligand PDBQT successfully generated: {ligand_pdbqt_out}")
print(f" Active rotatable bonds (torsions): {active_torsions}")

 Ligand PDBQT successfully generated: /content/single-dock/indinavir.pdbqt
 Active rotatable bonds (torsions): 28


# 4. Defining the Search Space (Grid Box)
Docking algorithms need boundaries to explore ligand conformations efficiently. Instead of searching the whole protein (**blind docking**), we define a **Grid Box** centered around the known binding cleft (**targeted docking**).

#### How to define the box:
* **Center ($x, y, z$):** The 3D centroid around catalytic residues (**Asp25/Asp25'**) and binding flap residues (**Val32, Ile47, Val82**).
* **Size ($x, y, z$):** Dimensions in Ångströms ($\text{Å}$). The box must be spacious enough to allow the ligand to rotate freely without touching the borders, but not excessively large to avoid scoring noise.

Use the interactive sliders below to inspect the pocket and adjust the search space.

In [21]:
from pathlib import Path
import py3Dmol
import ipywidgets as widgets
from ipywidgets import interact

# 1. Load cleaned receptor coordinates once
prot_path = Path("/content/single-dock/1hsg_prot.pdb")
prot_pdb_block = prot_path.read_text()

# Global configuration dictionary passed downstream to AutoDock Vina
grid_box_params = {
    "center_x": 16.0, "center_y": 25.0, "center_z": 4.0,
    "size_x": 22.0, "size_y": 22.0, "size_z": 22.0
}

# 2. Interactive 3D pocket visualizer
def show_docking_grid(cx, cy, cz, sx, sy, sz):
    """Render receptor cartoon, key catalytic residues, and search grid box."""
    grid_box_params["center_x"] = cx
    grid_box_params["center_y"] = cy
    grid_box_params["center_z"] = cz
    grid_box_params["size_x"] = sx
    grid_box_params["size_y"] = sy
    grid_box_params["size_z"] = sz

    viewer = py3Dmol.view(width=650, height=450)
    viewer.addModel(prot_pdb_block, "pdb")
    viewer.setStyle({"cartoon": {"color": "spectrum"}})

    # Highlight catalytic Asp25 and active site residues
    active_residues = [25, 32, 47, 82]
    viewer.addStyle(
        {"resi": active_residues},
        {"stick": {"colorscheme": "greenCarbon", "radius": 0.25}}
    )

    # Render bounding search box
    viewer.addBox({
        "center": {"x": cx, "y": cy, "z": cz},
        "dimensions": {"w": sx, "h": sy, "d": sz},
        "color": "blue",
        "opacity": 0.4
    })

    viewer.zoomTo({"resi": active_residues})
    viewer.setBackgroundColor("0x1e1e1e")
    viewer.show()

# 3. Interactive controls centered on the 1HSG active site
interact(
    show_docking_grid,
    cx=widgets.FloatSlider(min=-50, max=50, step=0.5, value=16.0, description="Center X"),
    cy=widgets.FloatSlider(min=-50, max=50, step=0.5, value=25.0, description="Center Y"),
    cz=widgets.FloatSlider(min=-50, max=50, step=0.5, value=4.0, description="Center Z"),
    sx=widgets.FloatSlider(min=10, max=40, step=1.0, value=22.0, description="Size X"),
    sy=widgets.FloatSlider(min=10, max=40, step=1.0, value=22.0, description="Size Y"),
    sz=widgets.FloatSlider(min=10, max=40, step=1.0, value=22.0, description="Size Z")
);

interactive(children=(FloatSlider(value=16.0, description='Center X', max=50.0, min=-50.0, step=0.5), FloatSli…

# 5. AutoDock Vina Configuration
To run AutoDock Vina, we write a configuration file defining the input structures, search volume, and simulation parameters.

#### Key Simulation Parameters:
* **`exhaustiveness` (Default: 8, here: 16):** Controls how thoroughly the algorithm searches conformational space. Higher values reduce the risk of getting trapped in local energy minima, which is especially important for flexible ligands like Indinavir.
* **`num_modes` (9):** The maximum number of top-scoring binding poses to save.
* **`energy_range` (3 kcal/mol):** Excludes poses with binding energies that differ by more than 3 kcal/mol from the best (top-ranked) mode.

In [29]:
from pathlib import Path

# 1. Setup workspace path
singlepath = Path("/content/single-dock")

# 2. Configuration parameters (without the deprecated 'log' option)
config_data = {
    "receptor": "1hsg_prot.pdbqt",
    "ligand": "indinavir.pdbqt",
    "out": "docking_output.pdbqt",
    "center_x": grid_box_params.get("center_x", 16.0),
    "center_y": grid_box_params.get("center_y", 25.0),
    "center_z": grid_box_params.get("center_z", 4.0),
    "size_x": grid_box_params.get("size_x", 22.0),
    "size_y": grid_box_params.get("size_y", 22.0),
    "size_z": grid_box_params.get("size_z", 22.0),
    "exhaustiveness": 16,
    "num_modes": 9,
    "energy_range": 3
}

# 3. Format config file
config_content = f"""# AutoDock Vina Configuration File
receptor = {config_data['receptor']}
ligand = {config_data['ligand']}
out = {config_data['out']}

# Grid Box Center
center_x = {config_data['center_x']}
center_y = {config_data['center_y']}
center_z = {config_data['center_z']}

# Grid Box Dimensions (Å)
size_x = {config_data['size_x']}
size_y = {config_data['size_y']}
size_z = {config_data['size_z']}

# Search Settings
exhaustiveness = {config_data['exhaustiveness']}
num_modes = {config_data['num_modes']}
energy_range = {config_data['energy_range']}
"""

# 4. Save to disk
config_file = singlepath / "config_singledock.txt"
config_file.write_text(config_content.strip())

print(f" Clean configuration file generated: {config_file.name}")

 Clean configuration file generated: config_singledock.txt


# 6. Running the Molecular Docking Simulation
Now we execute AutoDock Vina using our configuration file.

We pipe the terminal output through `tee docking.log` so you can monitor progress live while logging stdout. Vina's internal summary will independently write to `vina_internal.log`.

In [30]:
%%bash
# Navigate to workspace and run Vina
cd /content/single-dock
vina --config config_singledock.txt | tee docking.log

AutoDock Vina v1.2.7
#################################################################
# If you used AutoDock Vina in your work, please cite:          #
#                                                               #
# J. Eberhardt, D. Santos-Martins, A. F. Tillack, and S. Forli  #
# AutoDock Vina 1.2.0: New Docking Methods, Expanded Force      #
# Field, and Python Bindings, J. Chem. Inf. Model. (2021)       #
# DOI 10.1021/acs.jcim.1c00203                                  #
#                                                               #
# O. Trott, A. J. Olson,                                        #
# AutoDock Vina: improving the speed and accuracy of docking    #
# with a new scoring function, efficient optimization and       #
# multithreading, J. Comp. Chem. (2010)                         #
# DOI 10.1002/jcc.21334                                         #
#                                                               #
# Please see https://github.com/ccsb-scripps/AutoDock-V

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


### 6.1 Docking Results: Binding Affinity Table
AutoDock Vina ranks the predicted binding modes by **Binding Affinity** (in $\text{kcal/mol}$).

* **More negative values:** Indicate stronger, more favorable binding.
* **RMSD columns:** Show the structural difference compared to the top-scoring mode (Mode 1).

In [33]:
import pandas as pd
from pathlib import Path

# 1. Read Vina execution log
log_file = Path("/content/single-dock/docking.log")
log_lines = log_file.read_text().splitlines()

# 2. Parse score lines
data = []
for line in log_lines:
    parts = line.split()
    # Check if the line starts with a mode index (1 to 9)
    if parts and parts[0].isdigit() and 1 <= int(parts[0]) <= 9:
        mode = int(parts[0])
        affinity = float(parts[1])
        rmsd_lb = float(parts[2])
        # Clean up the 4th column in case a warning message attached to it
        rmsd_ub_raw = parts[3].split("/")[0]
        rmsd_ub = float(rmsd_ub_raw)
        data.append([mode, affinity, rmsd_lb, rmsd_ub])

# 3. Display formatted summary table
if data:
    df = pd.DataFrame(data, columns=["Mode", "Affinity (kcal/mol)", "RMSD l.b.", "RMSD u.b."])
    print("=== AutoDock Vina Docking Results ===")
    display(df.style.hide(axis="index").format({
        "Affinity (kcal/mol)": "{:.2f}",
        "RMSD l.b.": "{:.3f}",
        "RMSD u.b.": "{:.3f}"
    }))
else:
    print("Could not parse table. Make sure docking.log contains valid results.")

=== AutoDock Vina Docking Results ===


Mode,Affinity (kcal/mol),RMSD l.b.,RMSD u.b.
1,-10.89,0.000,0.000
2,-10.73,1.984,10.980
3,-10.50,1.948,10.220
4,-10.03,2.370,11.040
5,-9.99,2.239,10.740
6,-9.98,1.414,4.564
7,-9.93,2.133,11.060
8,-9.88,1.990,10.650
9,-9.71,2.142,10.300


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


### 6.2 Re-docking Validation: Extracting Native Crystal Pose
To test whether our docking simulation was accurate, we use a standard benchmark called **Re-docking Validation**:
* We extract the experimentally resolved coordinates of Indinavir (**MK1**) directly from the original crystal structure `1hsg.pdb` (our ground truth).
* We split the combined docking output (`docking_output.pdbqt`) into individual files using Scripps' official `vina_split` utility so we can isolate the top-ranked prediction (**Pose 1**).

In [34]:
from pathlib import Path

# 1. Setup paths
singlepath = Path("/content/single-dock")
raw_pdb = Path("1hsg.pdb")
xtal_ligand_pdb = singlepath / "xtal_ligand.pdb"

# 2. Extract crystal ligand using strict PDB column slicing (columns 18-20 for residue name)
with open(raw_pdb, "r") as infile, open(xtal_ligand_pdb, "w") as outfile:
    for line in infile:
        if line.startswith(("HETATM", "ATOM")):
            res_name = line[17:20].strip()
            if res_name == "MK1":
                outfile.write(line)
    outfile.write("END\n")

print(f"Native crystal ligand extracted: {xtal_ligand_pdb.name}")

Native crystal ligand extracted: xtal_ligand.pdb


In [35]:
%%bash
# Split multi-model PDBQT into individual pose files
cd /content/single-dock
vina_split --input docking_output.pdbqt

AutoDock Vina PDBQT Split v1.2.7
Prefix for ligands will be docking_output_ligand_
Prefix for flexible side chains will be docking_output_flex_


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


### 6.3 3D Superposition: Predicted Pose vs. Crystal Structure
Let's visually compare the predicted binding mode against experimental reality inside the catalytic pocket:
* **HIV-1 Protease Receptor:** Translucent cartoon (`opacity: 0.55`) so internal pocket interactions remain visible.
* **Predicted Pose 1 (AutoDock Vina):** <span style="color:#00FF00">**Green sticks**</span>.
* **Experimental Crystal Pose (Ground Truth):** <span style="color:#FF00FF">**Magenta sticks**</span>.

In [42]:
from pathlib import Path
import py3Dmol

singlepath = Path("/content/single-dock")

# 1. Read standard PDB files
prot_str = (singlepath / "1hsg_prot.pdb").read_text()
docked_pdb_str = (singlepath / "docking_output_ligand_1.pdb").read_text()
xtal_pdb_str = (singlepath / "xtal_ligand.pdb").read_text()

# 2. Setup viewer
viewer = py3Dmol.view(width=750, height=500)
viewer.setBackgroundColor("#1e1e1e")

# Receptor (Translucent cartoon)
viewer.addModel(prot_str, "pdb")
viewer.setStyle({"model": 0}, {"cartoon": {"color": "spectrum", "opacity": 0.4}})

# Docked Pose (Thick Green Sticks)
viewer.addModel(docked_pdb_str, "pdb")
viewer.setStyle({"model": 1}, {"stick": {"colorscheme": "greenCarbon", "radius": 0.25}})

# Crystal Ligand (Thinner Magenta Sticks)
viewer.addModel(xtal_pdb_str, "pdb")
viewer.setStyle({"model": 2}, {"stick": {"colorscheme": "magentaCarbon", "radius": 0.15}})

# Zoom directly onto the active pocket
viewer.zoomTo({"model": 2})
viewer.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

### 6.4 Quantitative Validation: Heavy-Atom Symmetry-Corrected RMSD
To quantitatively evaluate our docking prediction against the experimental ground truth, we compute the Root-Mean-Square Deviation (RMSD) across all heavy (non-hydrogen) atoms.

* **Challenge:** Simple coordinate subtraction fails because atom order and chemical symmetry differ between structures.
* **Solution:** We use `spyrmsd`, which employs the Hungarian algorithm to perform graph-isomorphic, symmetry-aware atomic mapping.
* **Benchmark Standard:** An $\text{RMSD} \le 2.0\text{ \AA}$ indicates a successful docking protocol.

In [38]:
!pip -q install spyrmsd

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 32.4 MB/s eta 0:00:00


In [48]:
from pathlib import Path
from spyrmsd import io, rmsd

singlepath = Path("/content/single-dock")
pose_pdbqt = singlepath / "docking_output_ligand_1.pdbqt"
pose_pdb = singlepath / "docking_output_ligand_1.pdb"
xtal_pdb = singlepath / "xtal_ligand.pdb"

# 1. Convert docked PDBQT to standard PDB format so RDKit/spyrmsd can parse it
!obabel -ipdbqt {pose_pdbqt} -opdb -O {pose_pdb} -h > /dev/null

# 2. Load molecules into spyrmsd using the standard PDB reader
mol_dock = io.loadmol(str(pose_pdb))
mol_xtal = io.loadmol(str(xtal_pdb))

# 3. Strip hydrogens to evaluate heavy-atom RMSD strictly
mol_dock.strip()
mol_xtal.strip()

# 4. Extract atomic properties for symmetry-aware graph matching
coords_dock = mol_dock.coordinates
coords_xtal = mol_xtal.coordinates
anum_dock = mol_dock.atomicnums
anum_xtal = mol_xtal.atomicnums
adj_dock = mol_dock.adjacency_matrix
adj_xtal = mol_xtal.adjacency_matrix

# 5. Compute symmetry-corrected RMSD (Hungarian matching)
val_rmsd = rmsd.symmrmsd(
    coords_xtal,
    coords_dock,
    anum_xtal,
    anum_dock,
    adj_xtal,
    adj_dock,
    minimize=False
)

# 6. Report validation benchmark
print(f" Symmetry-Corrected Heavy-Atom RMSD: {val_rmsd:.2f} Å")
if val_rmsd <= 2.0:
    print(" Benchmark Successful: Pose 1 accurately reproduces the crystal mode (RMSD ≤ 2.0 Å)!")
else:
    print(f"RMSD = {val_rmsd:.2f} Å: Check visual alignment in the 3D viewer.")

1 molecule converted
 Symmetry-Corrected Heavy-Atom RMSD: 0.66 Å
 Benchmark Successful: Pose 1 accurately reproduces the crystal mode (RMSD ≤ 2.0 Å)!


### 7. Summary & Conclusion
In this tutorial, we established a complete, modern molecular docking pipeline in Python:

1. **Receptor Preparation:** Fetched the HIV-1 protease dimer (`1HSG`), stripped crystallographic waters/heteroatoms, and parameterized partial charges using `PDB2PQR` and `OpenBabel`.
2. **Ligand Parameterization:** Converted a 1D canonical SMILES of Indinavir into a 3D conformer (`ETKDGv3` / `MMFF94`) and generated its rotatable-bond topology with `Meeko`.
3. **Grid Space Definition:** Interactively centered the search space around the catalytic `Asp25` dyad inside `py3Dmol`.
4. **Docking Engine:** Simulated binding poses using `AutoDock Vina v1.2.7`, achieving an affinity of **-10.89 kcal/mol** for Mode 1.
5. **Validation:** Confirmed structural fidelity against the native crystal pose with an outstanding **RMSD of 0.66 Å**.

In [47]:
import shutil
from pathlib import Path
from google.colab import files

# 1. Compress the results directory
results_dir = Path("/content/single-dock")
zip_filename = Path("/content/single_dock_results.zip")

print("Compressing docking workspace...")
shutil.make_archive("/content/single_dock_results", "zip", results_dir)

# 2. Trigger browser download
print("Initiating download...")
files.download(str(zip_filename))

Compressing docking workspace...
Initiating download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>